# Notebook 3 - Correlation & Lag Analysis

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from scipy.signal import correlate, correlation_lags
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

In [ ]:
OUTPUT_PATH = Path(r"path/to/your/data")

panel = pl.read_parquet(OUTPUT_PATH / "panel_features.parquet")

print(f"Panel shape : {panel.shape}")
print(f"Date range  : {panel['date'].min()} → {panel['date'].max()}")
print(f"Countries   : {panel['lhg_country'].n_unique()}")

## Cross-correlation function (CCF) analysis

The CCF measures correlation between a Reddit signal at time t-lag
and bookings at time t, across a range of lags.

Computed this per country and then aggregate to find the
most common optimal lag across the dataset.

In [ ]:
def compute_ccf(x: np.ndarray, y: np.ndarray, max_lag: int = 42) -> tuple:
    """
    Compute normalised cross-correlation between x (Reddit signal)
    and y (bookings) for lags 0 to max_lag.
    Positive lag = x leads y (Reddit predicts bookings).
    Returns (lags, correlations).
    """
    # Remove NaNs
    mask = ~(np.isnan(x) | np.isnan(y))
    x, y = x[mask], y[mask]

    if len(x) < 30:
        return np.array([]), np.array([])

    # Standardise both series
    x = (x - x.mean()) / (x.std() + 1e-8)
    y = (y - y.mean()) / (y.std() + 1e-8)

    # Full cross-correlation
    corr = correlate(y, x, mode='full')
    lags = correlation_lags(len(y), len(x), mode='full')
    corr = corr / len(x)   # normalise

    # Keep only positive lags (Reddit leading bookings)
    mask_lags = (lags >= 0) & (lags <= max_lag)
    return lags[mask_lags], corr[mask_lags]


print("CCF function defined.")

In [ ]:
# Run CCF for all countries × key Reddit features

REDDIT_SIGNALS = {
    "engagement":           "total_engagement_score",
    "weighted_sentiment":   "total_weighted_sentiment",
    "sentiment_momentum":   "sentiment_momentum",
    "engagement_zscore":    "engagement_zscore",
    "consistent_sentiment": "consistent_sentiment_score",
    "eng_roll7":            "eng_score_roll7sum",
    "eng_roll28":           "eng_score_roll28sum",
}

BOOKING_TARGET = "pax"          # raw bookings
MAX_LAG = 42                    # test up to 6 weeks ahead
countries = panel["lhg_country"].unique().sort().to_list()

# Store results: {signal_name: {country: (lags, corrs)}}
ccf_results = {sig: {} for sig in REDDIT_SIGNALS}

for country in countries:
    df_c = (
        panel
        .filter(pl.col("lhg_country") == country)
        .sort("date")
        .to_pandas()
    )
    y = df_c[BOOKING_TARGET].values.astype(float)

    for sig_name, sig_col in REDDIT_SIGNALS.items():
        x = df_c[sig_col].values.astype(float)
        lags, corrs = compute_ccf(x, y, max_lag=MAX_LAG)
        if len(lags) > 0:
            ccf_results[sig_name][country] = (lags, corrs)

print(f"CCF computed for {len(countries)} countries × {len(REDDIT_SIGNALS)} signals")

In [ ]:
#Aggregate CCF across all countries (mean correlation per lag)

fig, axes = plt.subplots(len(REDDIT_SIGNALS), 1,
                          figsize=(14, 3 * len(REDDIT_SIGNALS)),
                          sharex=True)

agg_ccf = {}   # {signal: (lags, mean_corr, std_corr)}

for ax, (sig_name, _) in zip(axes, REDDIT_SIGNALS.items()):
    all_corrs = []
    for country, (lags, corrs) in ccf_results[sig_name].items():
        if len(corrs) == MAX_LAG + 1:
            all_corrs.append(corrs)

    if not all_corrs:
        continue

    corr_matrix = np.array(all_corrs)
    mean_corr = corr_matrix.mean(axis=0)
    std_corr  = corr_matrix.std(axis=0)
    lag_range = np.arange(MAX_LAG + 1)
    agg_ccf[sig_name] = (lag_range, mean_corr, std_corr)

    optimal_lag = lag_range[np.argmax(mean_corr)]
    peak_corr   = mean_corr.max()

    ax.plot(lag_range, mean_corr, linewidth=2, label=f"Mean corr (peak lag={optimal_lag}d)")
    ax.fill_between(lag_range,
                    mean_corr - std_corr,
                    mean_corr + std_corr,
                    alpha=0.2, label="±1 std")
    ax.axvline(optimal_lag, color='crimson', linestyle='--', linewidth=1)
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.set_title(f"{sig_name}  |  peak r={peak_corr:.3f} at lag={optimal_lag}d",
                 fontsize=11)
    ax.set_ylabel("Correlation")
    ax.legend(fontsize=8)

axes[-1].set_xlabel("Lag (days) — Reddit leads bookings →")
plt.suptitle("Cross-correlation: Reddit signals → Bookings (aggregated across all countries)",
             y=1.01, fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "ccf_all_signals.png", dpi=150, bbox_inches='tight')
plt.show()

print("Optimal lag summary")
for sig_name, (lags, mean_corr, _) in agg_ccf.items():
    opt_lag = lags[np.argmax(mean_corr)]
    peak    = mean_corr.max()
    print(f"  {sig_name:<25} optimal lag = {opt_lag:>2}d   peak r = {peak:.4f}")

## Per-country optimal lag

The aggregate view hides country-level variation.
Here we find the optimal lag per country for the best-performing signal.

In [ ]:
# Use engagement as the primary signal for per-country lag analysis
# (swap to whichever signal had the highest peak correlation above)
PRIMARY_SIGNAL = max(agg_ccf, key=lambda s: agg_ccf[s][1].max())
print(f"Best performing signal overall: {PRIMARY_SIGNAL}")

per_country_lag = []
for country, (lags, corrs) in ccf_results[PRIMARY_SIGNAL].items():
    if len(corrs) == 0:
        continue
    opt_lag  = lags[np.argmax(corrs)]
    peak_r   = corrs.max()
    per_country_lag.append({
        "country":     country,
        "optimal_lag": int(opt_lag),
        "peak_r":      round(float(peak_r), 4),
    })

lag_df = pd.DataFrame(per_country_lag).sort_values("peak_r", ascending=False)
print()
print("Per-country optimal lag (top 20 by correlation strength):")
print(lag_df.head(20).to_string(index=False))

In [ ]:
# Lag distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(lag_df["optimal_lag"], bins=range(0, MAX_LAG + 2),
             edgecolor='white', color='steelblue')
axes[0].set_title(f"Distribution of optimal lags ({PRIMARY_SIGNAL})")
axes[0].set_xlabel("Optimal lag (days)")
axes[0].set_ylabel("Number of countries")
axes[0].axvline(lag_df["optimal_lag"].median(), color='crimson',
                linestyle='--', label=f'Median = {lag_df["optimal_lag"].median():.0f}d')
axes[0].legend()

axes[1].scatter(lag_df["optimal_lag"], lag_df["peak_r"],
                alpha=0.7, s=60, color='steelblue')
# for _, row in lag_df.iterrows():
#     axes[1].annotate(row["country"],
#                      (row["optimal_lag"], row["peak_r"]),
#                      fontsize=7, alpha=0.8,
#                      xytext=(3, 3), textcoords='offset points')
axes[1].set_title("Optimal lag vs correlation strength per country")
axes[1].set_xlabel("Optimal lag (days)")
axes[1].set_ylabel("Peak correlation (r)")
axes[1].axhline(0, color='gray', linewidth=0.5)

plt.tight_layout()
plt.savefig(OUTPUT_PATH / "lag_distribution.png", dpi=150)
plt.show()

print(f"\nMedian optimal lag : {lag_df['optimal_lag'].median():.0f} days")
print(f"Mean optimal lag   : {lag_df['optimal_lag'].mean():.1f} days")
print(f"Countries with r>0 : {(lag_df['peak_r'] > 0).sum()} / {len(lag_df)}")

## Sentiment vs Consistent Sentiment

In [ ]:
sent_signals = {
    "sentiment":            "weighted_sentiment",
    "consistent_sentiment": "consistent_sentiment",
    "sentiment_momentum":   "sentiment_momentum",
}

fig, ax = plt.subplots(figsize=(14, 5))

colors = ["steelblue", "darkorange", "green"]
results_summary = []

for (sig_name, _), color in zip(sent_signals.items(), colors):
    if sig_name not in agg_ccf:
        continue
    lags, mean_corr, std_corr = agg_ccf[sig_name]
    opt_lag  = lags[np.argmax(mean_corr)]
    peak_r   = mean_corr.max()

    ax.plot(lags, mean_corr, linewidth=2.5,
            label=f"{sig_name}  (peak r={peak_r:.3f} @ lag={opt_lag}d)",
            color=color)
    ax.fill_between(lags, mean_corr - std_corr, mean_corr + std_corr,
                    alpha=0.1, color=color)

    results_summary.append({
        "signal": sig_name,
        "optimal_lag": int(opt_lag),
        "peak_r": round(float(peak_r), 4),
        "mean_r_all_lags": round(float(mean_corr.mean()), 4),
    })

ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
ax.set_xlabel("Lag (days)")
ax.set_ylabel("Mean correlation across countries")
ax.set_title("Sentiment vs Consistent Sentiment — predictive correlation with bookings")
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "sentiment_vs_consistent.png", dpi=150)
plt.show()

print("Comparison summary")
print(pd.DataFrame(results_summary).to_string(index=False))
print()
print("Interpretation:")
print("  Higher peak_r = stronger predictor of booking surges")
print("  Lower optimal_lag = faster customer reaction to Reddit signal")
print("  Higher mean_r_all_lags = more consistently predictive across all time horizons")

## Spike co-occurrence analysis

Alternate: instead of continuous correlation,
check whether Reddit engagement spikes (engagement_zscore > 2)
are followed by booking spikes within a given window.

When Reddit goes viral
about a destination, does it actually drive bookings in the next X days?

In [ ]:
ENGAGEMENT_SPIKE_THRESHOLD = 2.0
BOOKING_SPIKE_COL          = "pax_spike_positive"
WINDOWS_TO_TEST            = [7, 14, 21, 28]

# Convert to pandas for rolling window event analysis
panel_pd = panel.to_pandas().sort_values(["lhg_country", "date"])

cooccurrence_results = []

for country in countries:
    df_c = panel_pd[panel_pd["lhg_country"] == country].copy()
    df_c = df_c.set_index("date").sort_index()

    reddit_spike_dates = df_c[
        df_c["engagement_zscore"] > ENGAGEMENT_SPIKE_THRESHOLD
    ].index

    if len(reddit_spike_dates) == 0:
        continue

    for window in WINDOWS_TO_TEST:
        followed_by_booking_spike = 0
        for spike_date in reddit_spike_dates:
            # Check if any booking spike occurs in the next `window` days
            future = df_c.loc[
                (df_c.index > spike_date) &
                (df_c.index <= spike_date + pd.Timedelta(days=window))
            ]
            if future[BOOKING_SPIKE_COL].sum() > 0:
                followed_by_booking_spike += 1

        hit_rate = followed_by_booking_spike / len(reddit_spike_dates)
        cooccurrence_results.append({
            "country":             country,
            "window_days":         window,
            "reddit_spikes":       len(reddit_spike_dates),
            "followed_by_booking": followed_by_booking_spike,
            "hit_rate":            round(hit_rate, 3),
        })

cooc_df = pd.DataFrame(cooccurrence_results)

# Aggregate hit rate by window
print("Hit rate: P(booking spike within N days | Reddit spike)")
print(cooc_df.groupby("window_days")["hit_rate"].agg(["mean", "median", "std"]).round(3))

In [ ]:
# Visualise hit rates
hit_by_window = cooc_df.groupby("window_days")["hit_rate"].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Average hit rate by window
axes[0].bar(hit_by_window.index.astype(str),
            hit_by_window.values,
            color='steelblue', edgecolor='white')
axes[0].set_title("Average hit rate by follow-up window")
axes[0].set_xlabel("Days after Reddit spike")
axes[0].set_ylabel("P(booking spike follows)")
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

# Per-country hit rate heatmap (14-day window)
cooc_14 = cooc_df[cooc_df["window_days"] == 14].set_index("country")["hit_rate"]
cooc_14_sorted = cooc_14.sort_values(ascending=False)

axes[1].barh(cooc_14_sorted.index, cooc_14_sorted.values,
             color='steelblue', edgecolor='white')
# Replace country names with anonymized labels
axes[1].set_yticks(range(len(cooc_14_sorted.index)))
axes[1].set_yticklabels([f"Country {i+1}" for i in range(len(cooc_14_sorted.index))])
axes[1].tick_params(axis='y', labelsize=8)
axes[1].set_title("Hit rate per country (14-day window)")
axes[1].set_xlabel("P(booking spike within 14d | Reddit spike)")
axes[1].xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
axes[1].axvline(cooc_14.mean(), color='crimson', linestyle='--',
                label=f'Mean = {cooc_14.mean():.1%}')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_PATH / "spike_cooccurrence.png", dpi=150)
plt.show()

## Correlation heatmap across all signals and fixed lags

Using the pre-built lag features from notebook 2,
compute Pearson and Spearman correlations for each
signal × lag combination against bookings.

In [ ]:
# Build correlation matrix: signal × lag → r with pax

LAG_DAYS    = [7, 14, 21, 28]
BASE_FEATS  = [
    "total_engagement_score",
    "total_weighted_sentiment",
    "mean_sentiment",
    "eng_score_roll7sum",
    "eng_score_roll28sum",
    "sentiment_momentum",
    "engagement_momentum",
    "engagement_zscore",
    "consistent_sentiment_score",
    "positive_days_14d",
]

panel_pd_full = panel.to_pandas()
pax = panel_pd_full["pax"].values

corr_data = {}
pval_data = {}

for feat in BASE_FEATS:
    corr_data[feat] = {}
    pval_data[feat] = {}
    for lag in LAG_DAYS:
        col = f"{feat}_lag{lag}"
        if col not in panel_pd_full.columns:
            continue
        x = panel_pd_full[col].values
        # Drop rows where either is NaN
        mask = ~(np.isnan(x) | np.isnan(pax))
        if mask.sum() < 30:
            continue
        r, p = stats.spearmanr(x[mask], pax[mask])
        corr_data[feat][f"lag{lag}d"] = round(r, 4)
        pval_data[feat][f"lag{lag}d"] = p

corr_matrix = pd.DataFrame(corr_data).T
pval_matrix = pd.DataFrame(pval_data).T

print("Spearman correlation matrix (Reddit signal lag → bookings):")
print(corr_matrix.round(4))

In [ ]:
# Heatmap
fig, ax = plt.subplots(figsize=(10, 8))

# Mask non-significant correlations (p > 0.05)
sig_mask = pval_matrix > 0.05

sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".3f",
    cmap="RdBu_r",
    center=0,
    vmin=-0.3,
    vmax=0.3,
    linewidths=0.5,
    ax=ax,
    mask=sig_mask,          # grey out non-significant cells
    cbar_kws={"label": "Spearman r"}
)

# Overlay hatching on non-significant cells
sns.heatmap(
    corr_matrix,
    annot=False,
    cmap=["#f0f0f0"],
    center=0,
    linewidths=0.5,
    ax=ax,
    mask=~sig_mask,
    cbar=False,
    alpha=0.4
)

ax.set_title("Spearman correlation: Reddit features (lagged) → Daily bookings\n"
             "(greyed cells: p > 0.05, not significant)",
             fontsize=12)
ax.set_xlabel("Lag")
ax.set_ylabel("Reddit feature")
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "correlation_heatmap.png", dpi=150)
plt.show()

## Top correlated country-signal pairs

To get concrete examples.

In [ ]:
# Per-country Spearman correlation for primary signal at each lag 
PRIMARY_FEAT = "total_engagement_score"

per_country_corr = []
for country in countries:
    df_c = panel_pd_full[panel_pd_full["lhg_country"] == country]
    pax_c = df_c["pax"].values

    for lag in LAG_DAYS:
        col = f"{PRIMARY_FEAT}_lag{lag}"
        if col not in df_c.columns:
            continue
        x = df_c[col].values
        mask = ~(np.isnan(x) | np.isnan(pax_c))
        if mask.sum() < 20:
            continue
        r, p = stats.spearmanr(x[mask], pax_c[mask])
        per_country_corr.append({
            "country": country,
            "lag":     lag,
            "r":       round(float(r), 4),
            "p":       round(float(p), 4),
            "sig":     "*" if p < 0.05 else "",
        })

pc_df = pd.DataFrame(per_country_corr)

# Pivot to country × lag table
pivot = pc_df.pivot(index="country", columns="lag", values="r")
pivot.columns = [f"lag{c}d" for c in pivot.columns]

# Sort by best correlation across any lag
pivot["best_r"] = pivot.max(axis=1)
pivot = pivot.sort_values("best_r", ascending=False)

print(f"Per-country correlation — {PRIMARY_FEAT}")
print(pivot.round(4).to_string())

In [ ]:
# Heatmap: country × lag
plot_data = pivot.drop(columns="best_r")

fig, ax = plt.subplots(figsize=(8, max(8, len(pivot) * 0.4)))
sns.heatmap(
    plot_data,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    linewidths=0.5,
    ax=ax,
    cbar_kws={"label": "Spearman r"}
)
ax.set_yticks([])
ax.set_title(f"Spearman r: {PRIMARY_FEAT} (lagged) → pax\nper country",
             fontsize=12)
ax.set_xlabel("Lag")
ax.set_ylabel("Countries")
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "correlation_per_country.png", dpi=150)
plt.show()

In [ ]:
# Visual deep-dive for top country
top_country = pivot.index[0]
best_lag    = int(pivot.loc[top_country].drop("best_r", errors="ignore").idxmax().replace("lag","").replace("d",""))

print(f"Top country: {top_country} | Best lag: {best_lag}d")

df_top = (
    panel
    .filter(pl.col("lhg_country") == top_country)
    .sort("date")
    .to_pandas()
    .set_index("date")
)

lag_col = f"{PRIMARY_FEAT}_lag{best_lag}"

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

ax1 = axes[0]
ax1.plot(df_top["pax"], color="steelblue", linewidth=1, label="Daily bookings")
ax1.set_title(f"{top_country} — Bookings vs Reddit engagement (lag={best_lag}d)")
ax1.set_ylabel("Passengers", color="steelblue")
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

ax1b = ax1.twinx()
ax1b.plot(df_top[lag_col], color="darkorange", linewidth=1,
          alpha=0.7, label=f"Reddit engagement (lag={best_lag}d)")
ax1b.set_ylabel("Engagement score (lagged)", color="darkorange")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax1b.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=9)

axes[1].plot(df_top["pax_zscore"], color="steelblue", linewidth=0.8, label="Booking z-score")
axes[1].axhline(2, color='crimson', linestyle='--', linewidth=1, label="Spike threshold")
axes[1].axhline(-2, color='crimson', linestyle='--', linewidth=1)
axes[1].axhline(0, color='gray', linewidth=0.5)
axes[1].set_title("Booking z-score (spike detection)")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_PATH / f"deep_dive_{top_country}.png", dpi=150)
plt.show()

In [ ]:
# Deseasonalized correlation analysis
# The high r values in the heatmap above are partially inflated by shared
# seasonality — both Reddit engagement and bookings peak in summer.
# Here we remove the weekly seasonal component from bookings per country
# and re-run the correlation.

from statsmodels.tsa.seasonal import STL

# REDDIT_FEATS_TO_TEST = [
#     "total_engagement_score",
#     "eng_score_roll7sum",
#     "eng_score_roll28sum",
#     "consistent_sentiment_score",
#     "positive_days_14d",
#     "positive_days_14d_detrended",
#     "mean_sentiment",
# ]

REDDIT_FEATS_TO_TEST = [
    "total_engagement_score",
    "eng_score_roll7sum",
    "eng_score_roll28sum",
    "consistent_sentiment_score",
    "mean_sentiment",
]

deseas_corr_data = {feat: {} for feat in REDDIT_FEATS_TO_TEST}
panel_pd_full = panel.to_pandas()

for country in countries:
    df_c = (
        panel_pd_full[panel_pd_full["lhg_country"] == country]
        .sort_values("date")
        .set_index("date")
    )

    # Need at least 2 full weeks for STL with period=7
    if len(df_c) < 28:
        continue

    # Deseasonalize bookings for this country
    try:
        pax_series = df_c["pax"].asfreq("D").fillna(0)
        stl = STL(pax_series, period=7, robust=True)
        stl_fit = stl.fit()
        pax_deseas = (pax_series - stl_fit.seasonal).values
    except Exception:
        continue

    for feat in REDDIT_FEATS_TO_TEST:
        for lag in LAG_DAYS:
            col = f"{feat}_lag{lag}"
            if col not in df_c.columns:
                continue
            x = df_c[col].values.astype(float)
            mask = ~(np.isnan(x) | np.isnan(pax_deseas))
            if mask.sum() < 20:
                continue
            r, p = stats.spearmanr(x[mask], pax_deseas[mask])
            key = f"lag{lag}d"
            if key not in deseas_corr_data[feat]:
                deseas_corr_data[feat][key] = []
            deseas_corr_data[feat][key].append(r)

# Build mean correlation matrix
deseas_corr_matrix = pd.DataFrame({
    feat: {lag: np.mean(vals) for lag, vals in lag_dict.items()}
    for feat, lag_dict in deseas_corr_data.items()
}).T

print("Deseasonalized Spearman correlation matrix:")
print(deseas_corr_matrix.round(4))

In [ ]:
# Side by side comparison: raw vs deseasonalized
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Shared features and lags for both heatmaps
shared_feats = [f for f in REDDIT_FEATS_TO_TEST if f in corr_matrix.index]
shared_lags  = ["lag7d", "lag14d", "lag21d", "lag28d"]

raw_subset = corr_matrix.loc[shared_feats, shared_lags]
des_subset = deseas_corr_matrix.loc[shared_feats, shared_lags]

# Use same scale for fair comparison
vmax = max(raw_subset.values.max(), des_subset.values.max())
vmin = min(raw_subset.values.min(), des_subset.values.min())

for ax, data, title in zip(
    axes,
    [raw_subset, des_subset],
    ["Raw bookings (Spearman r)", "Deseasonalized bookings (Spearman r)"]
):
    sns.heatmap(
        data,
        annot=True,
        fmt=".3f",
        cmap="RdBu_r",
        center=0,
        vmin=vmin,
        vmax=vmax,
        linewidths=0.5,
        ax=ax,
        cbar_kws={"label": "Spearman r"}
    )
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Lag")
    ax.set_ylabel("")

plt.suptitle(
    "Effect of deseasonalization on Reddit → Booking correlations\n"
    "Drop in r = portion explained by shared seasonality (not genuine signal)",
    fontsize=12
)
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "correlation_deseasonalized_comparison.png", dpi=150)
plt.show()

print("\nDrop in correlation after deseasonalization")
print("(Large drops = correlation was mostly seasonal confound)")
print((raw_subset - des_subset).round(4).to_string())

## Save outputs

In [ ]:
# Save per-country correlation table
pivot.reset_index().to_csv(OUTPUT_PATH / "correlation_per_country.csv", index=False)

# Save optimal lag per country
lag_df.to_csv(OUTPUT_PATH / "optimal_lag_per_country.csv", index=False)

# Save spike co-occurrence results
cooc_df.to_csv(OUTPUT_PATH / "spike_cooccurrence.csv", index=False)

# Save full correlation matrix
corr_matrix.to_csv(OUTPUT_PATH / "correlation_matrix.csv")

deseas_corr_matrix.to_csv(OUTPUT_PATH / "correlation_matrix_deseasonalized.csv")


print("Saved:")
print("  outputs/correlation_per_country.csv")
print("  outputs/optimal_lag_per_country.csv")
print("  outputs/spike_cooccurrence.csv")
print("  outputs/correlation_matrix.csv")
print("  outputs/correlation_matrix_deseasonalized.csv")



In [ ]:
print("Findings for future")
print(f"  Best signal overall : {PRIMARY_SIGNAL}")
print(f"  Median optimal lag  : {lag_df['optimal_lag'].median():.0f} days")
print(f"  Top country         : {top_country} (r={pivot.loc[top_country, 'best_r']:.3f})")